# Dataset Preparation — Task 3 Trip Request Forecasting

Adapted from `dataset/data_vis.ipynb`. Differences vs. the original notebook:

1. **`N_ZONES = 263`** — the ablation framework fixes the OD geometry at 263 zones.
   Zones 264/265 (`Unknown`) are dropped during cleaning.
2. **Full month span** — downloads every month covering the train / val / test / OOD
   splits so a single continuous slot index can be built.
3. **Chronological splits** — after building the tensor, computes the
   `[start, end)` index ranges for each split and writes `splits.json`.

Split plan:

| Split | Months | Purpose |
|-------|--------|---------|
| Train | 2023-07 → 2024-09 | model fitting + all frozen statistics |
| Val   | 2024-10 → 2024-11 | early stopping / model selection |
| Test  | 2024-12           | in-distribution evaluation |
| OOD   | 2025-03 → 2025-04 | out-of-distribution robustness |

2025-01/02 are downloaded too (kept *unassigned*) so the slot index stays
continuous and lag features work across the OOD boundary.

> **Memory note:** the dense tensor is `(T, 263, 263)` int32. The full span is
> ~22 months ≈ 64k slots ≈ 18 GB in RAM. On a memory-constrained machine,
> shorten `YEAR_MONTHS` (and adjust the split dates) to run a pilot.

In [1]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd

# This notebook lives in scripts/; make the project root importable.
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data.splits import chronological_split, DEFAULT_SPLITS

# ---- configuration -------------------------------------------------------
N_ZONES = 263            # taxi zones 1..263; 264/265 (Unknown) are dropped
SLOT_MINUTES = 15

DATA_DIR = ROOT / "dataset" / "data"
OUT_DIR = ROOT / "dataset" / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Full month span covering train + val + test + OOD (2025-01/02 included so the
# slot index is continuous). Trim this list to run a smaller pilot.
YEAR_MONTHS = [d.strftime("%Y-%m")
               for d in pd.date_range("2023-07-01", "2025-04-01", freq="MS")]
print(f"{len(YEAR_MONTHS)} months: {YEAR_MONTHS[0]} ... {YEAR_MONTHS[-1]}")
print("split dates:", json.dumps(DEFAULT_SPLITS, indent=2))

22 months: 2023-07 ... 2025-04
split dates: {
  "train_start": "2023-07-01",
  "train_end": "2024-10-01",
  "val_end": "2024-12-01",
  "test_end": "2025-01-01",
  "ood_start": "2025-03-01",
  "ood_end": "2025-05-01"
}


## 1. Download raw HVFHV parquet files

Existing files are skipped. Each file is ~450–500 MB.

In [2]:
import wget

for m in YEAR_MONTHS:
    out = DATA_DIR / f"fhvhv_tripdata_{m}.parquet"
    if out.exists():
        print(f"existed: {out.name}")
        continue
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_{m}.parquet"
    print(f"downloading: {out.name}")
    wget.download(url, out=str(DATA_DIR))
    print()

downloading: fhvhv_tripdata_2023-07.parquet

downloading: fhvhv_tripdata_2023-08.parquet

downloading: fhvhv_tripdata_2023-09.parquet

downloading: fhvhv_tripdata_2023-10.parquet

downloading: fhvhv_tripdata_2023-11.parquet

downloading: fhvhv_tripdata_2023-12.parquet

existed: fhvhv_tripdata_2024-01.parquet
existed: fhvhv_tripdata_2024-02.parquet
existed: fhvhv_tripdata_2024-03.parquet
downloading: fhvhv_tripdata_2024-04.parquet

downloading: fhvhv_tripdata_2024-05.parquet

downloading: fhvhv_tripdata_2024-06.parquet

downloading: fhvhv_tripdata_2024-07.parquet

downloading: fhvhv_tripdata_2024-08.parquet

downloading: fhvhv_tripdata_2024-09.parquet

downloading: fhvhv_tripdata_2024-10.parquet

downloading: fhvhv_tripdata_2024-11.parquet

downloading: fhvhv_tripdata_2024-12.parquet

downloading: fhvhv_tripdata_2025-01.parquet

downloading: fhvhv_tripdata_2025-02.parquet

downloading: fhvhv_tripdata_2025-03.parquet

downloading: fhvhv_tripdata_2025-04.parquet



## 2. Pipeline functions

Same as `data_vis.ipynb` except `N_ZONES = 263`: `load_and_clean` now keeps only
zone IDs in `1..263`, so zones 264/265 are excluded from every artifact.

In [3]:
SLOTS_PER_DAY = 24 * 60 // SLOT_MINUTES


def load_and_clean(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path, columns=[
        "request_datetime", "pickup_datetime",
        "PULocationID", "DOLocationID",
    ])
    # request time, falling back to pickup time when missing
    df["t_req"] = df["request_datetime"].fillna(df["pickup_datetime"])
    df = df.dropna(subset=["t_req", "PULocationID", "DOLocationID"])
    # keep valid zone IDs only (1..263)
    df = df[
        df["PULocationID"].between(1, N_ZONES)
        & df["DOLocationID"].between(1, N_ZONES)
    ]
    # clip rows to the file's own calendar month
    file_month = pd.to_datetime(path.stem.replace("fhvhv_tripdata_", "") + "-01")
    next_month = file_month + pd.offsets.MonthBegin(1)
    df = df[(df["t_req"] >= file_month) & (df["t_req"] < next_month)]
    return df[["t_req", "PULocationID", "DOLocationID"]]


def aggregate_to_15min_od(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["slot_start"] = df["t_req"].dt.floor(f"{SLOT_MINUTES}min")
    return (
        df.groupby(["slot_start", "PULocationID", "DOLocationID"], observed=True)
          .size()
          .reset_index(name="count")
    )


def build_continuous_slot_index(start, end) -> pd.DatetimeIndex:
    """Continuous 15-min index, left-closed / right-open."""
    return pd.date_range(start=start, end=end,
                         freq=f"{SLOT_MINUTES}min", inclusive="left")


def to_dense_tensor(agg: pd.DataFrame, slot_index: pd.DatetimeIndex) -> np.ndarray:
    """Expand the sparse COO aggregate into a dense (T, N, N) int32 tensor."""
    T, N = len(slot_index), N_ZONES
    tensor = np.zeros((T, N, N), dtype=np.int32)
    slot_to_t = {ts: i for i, ts in enumerate(slot_index)}
    agg = agg[agg["slot_start"].isin(slot_to_t)]
    t_idx = agg["slot_start"].map(slot_to_t).to_numpy()
    o_idx = agg["PULocationID"].to_numpy() - 1   # 1-based ID -> 0-based index
    d_idx = agg["DOLocationID"].to_numpy() - 1
    np.add.at(tensor, (t_idx, o_idx, d_idx), agg["count"].to_numpy())
    return tensor

## 3. Run the pipeline and save artifacts

Writes `od_15min_long.parquet` (sparse COO) and `od_15min_tensor.npz`
(dense `(T, 263, 263)` tensor + epoch-ns `slot_starts`).

In [4]:
all_agg = []
for ym in YEAR_MONTHS:
    path = DATA_DIR / f"fhvhv_tripdata_{ym}.parquet"
    df = load_and_clean(path)
    agg = aggregate_to_15min_od(df)
    print(f"{ym}: {len(df):>12,} records -> {len(agg):>10,} (slot,O,D) tuples")
    all_agg.append(agg)

full_agg = pd.concat(all_agg, ignore_index=True)
full_agg = full_agg.groupby(
    ["slot_start", "PULocationID", "DOLocationID"], as_index=False
).agg({"count": "sum"})

long_path = OUT_DIR / "od_15min_long.parquet"
full_agg.to_parquet(long_path, index=False)
print(f"\n[saved] {long_path}  shape={full_agg.shape}")

start = full_agg["slot_start"].min()
end = full_agg["slot_start"].max() + pd.Timedelta(minutes=SLOT_MINUTES)
slot_index = build_continuous_slot_index(start, end)

tensor = to_dense_tensor(full_agg, slot_index)
slot_starts = slot_index.astype(np.int64).to_numpy()   # epoch-ns int64
np.savez_compressed(OUT_DIR / "od_15min_tensor.npz",
                    tensor=tensor, slot_starts=slot_starts)
print(f"[saved] {OUT_DIR / 'od_15min_tensor.npz'}")
print(f"        tensor {tensor.shape} {tensor.dtype}, "
      f"total trips {tensor.sum():,}")

2023-07:   18,334,252 records -> 11,638,595 (slot,O,D) tuples
2023-08:   17,549,821 records -> 11,373,479 (slot,O,D) tuples
2023-09:   19,044,286 records -> 11,683,595 (slot,O,D) tuples
2023-10:   19,340,315 records -> 11,928,202 (slot,O,D) tuples
2023-11:   18,449,151 records -> 11,371,843 (slot,O,D) tuples
2023-12:   19,632,229 records -> 11,960,644 (slot,O,D) tuples
2024-01:   18,921,450 records -> 11,514,327 (slot,O,D) tuples
2024-02:   18,603,806 records -> 11,320,434 (slot,O,D) tuples
2024-03:   20,445,164 records -> 12,293,499 (slot,O,D) tuples
2024-04:   18,925,432 records -> 11,621,570 (slot,O,D) tuples
2024-05:   19,829,777 records -> 12,180,100 (slot,O,D) tuples
2024-06:   19,254,854 records -> 11,918,633 (slot,O,D) tuples
2024-07:   18,372,438 records -> 11,664,797 (slot,O,D) tuples
2024-08:   18,316,187 records -> 11,668,988 (slot,O,D) tuples
2024-09:   18,358,471 records -> 11,500,868 (slot,O,D) tuples
2024-10:   19,129,753 records -> 11,939,030 (slot,O,D) tuples
2024-11:

## 4. Chronological splits

Maps the `DEFAULT_SPLITS` calendar onto `[start, end)` index ranges over the
slot index, then persists `splits.json`. All downstream feature fitting and
scaling must use the **train** range only.

In [5]:
splits = chronological_split(
    slot_starts,
    train_end=DEFAULT_SPLITS["train_end"],
    val_end=DEFAULT_SPLITS["val_end"],
    test_end=DEFAULT_SPLITS["test_end"],
    ood_start=DEFAULT_SPLITS["ood_start"],
    ood_end=DEFAULT_SPLITS["ood_end"],
    train_start=DEFAULT_SPLITS["train_start"],
)

for name, (s, e) in splits.items():
    if e > s:
        rng = f"{slot_index[s]} .. {slot_index[e - 1]}"
    else:
        rng = "(empty)"
    print(f"{name:6s} idx [{s:>6}, {e:>6})  {e - s:>6} slots  {rng}")

splits_meta = {
    "slot_minutes": SLOT_MINUTES,
    "n_zones": N_ZONES,
    "n_slots": int(len(slot_index)),
    "slot_start_first": str(slot_index[0]),
    "slot_start_last": str(slot_index[-1]),
    "split_dates": DEFAULT_SPLITS,
    "split_idx": {k: [int(s), int(e)] for k, (s, e) in splits.items()},
}
with open(OUT_DIR / "splits.json", "w") as f:
    json.dump(splits_meta, f, indent=2)
print(f"\n[saved] {OUT_DIR / 'splits.json'}")

train  idx [     0,  43968)   43968 slots  2023-07-01 00:00:00 .. 2024-09-30 23:45:00
val    idx [ 43968,  49824)    5856 slots  2024-10-01 00:00:00 .. 2024-11-30 23:45:00
test   idx [ 49824,  52800)    2976 slots  2024-12-01 00:00:00 .. 2024-12-31 23:45:00
ood    idx [ 58464,  64320)    5856 slots  2025-03-01 00:00:00 .. 2025-04-30 23:45:00

[saved] e:\Workspace\NASA_ULI\dataset\output\splits.json


## 5. Sanity checks

Quick guards before the data is consumed by the training pipeline.

In [6]:
tr, va, te, ood = (splits[k] for k in ("train", "val", "test", "ood"))

assert tensor.shape[1] == tensor.shape[2] == N_ZONES, "tensor must be (T,263,263)"
assert (tensor >= 0).all(), "negative OD counts"
assert tr[1] == va[0] and va[1] == te[0], "train/val/test must be contiguous"
assert tr[1] <= ood[0], "OOD must start after train"
assert all(e > s for s, e in (tr, va, te, ood)), "a split is empty -- check months"

# the long table total must match the dense tensor total
assert int(full_agg["count"].sum()) == int(tensor.sum()), "long/dense mismatch"

print("all sanity checks passed")
print(f"  train slots: {tr[1] - tr[0]:,}")
print(f"  val   slots: {va[1] - va[0]:,}")
print(f"  test  slots: {te[1] - te[0]:,}")
print(f"  ood   slots: {ood[1] - ood[0]:,}")

all sanity checks passed
  train slots: 43,968
  val   slots: 5,856
  test  slots: 2,976
  ood   slots: 5,856
